# TMDB Movie ID Ingestion

Importing packages

In [1]:
import os
from dotenv import load_dotenv
import requests
import json
import pandas as pd
from datetime import datetime
import calendar
import time

Importing credentials

In [2]:
load_dotenv()

tmdb_token = os.getenv("tmdb_bearer_token")
omdb_api_key = os.getenv("omdb_api_key")

Functions to Get TMDB IDs

In [6]:
def fetch_page_data(year, month, page):
  """Job is to fetch and return a pages data
  """
  first_day= 1
  last_day= calendar.monthrange(year, month)[1]
  #calendar.monthrage() returns the day of the week that a month starts on and the number of days in that month inside a tuple
  # having [1] gets me the second item which is the number of days and I can use that number as the last day in the month
  
  tmdb_discover_endpoint = 'https://api.themoviedb.org/3/discover/movie'
  params = {'with_original_language':'en',
          'primary_release_date.gte':f'{year}-{month:02d}-{first_day:02d}',
          'primary_release_date.lte':f'{year}-{month:02d}-{last_day}',
          'include_adult':'false',
          'include_video':'false',
          'sort_by':'primary_release_date.asc',
          'page':page}

  headers = {'accept':'application/json',
            'Authorization':f'Bearer {tmdb_token}'
            }
  response = requests.get(url=tmdb_discover_endpoint, params=params, headers=headers)

  data = response.json()
  return data

In [7]:
def extract_key_data(data):
  """Job is to take the json that is stored in my data variable and extract the:
  - movie ids
  - total_results
  - total_pages
  -current_page
  """
  tmdb_movie_ids = [movie['id']for movie in data['results']] # stores the movie ids from a page in a list
  total_pages = data['total_pages'] # stores the amount of pages that are in the search range
  total_results = data['total_results'] # stores the amount of results that are in the search range
  current_page = data['page'] #stores the current page thats data is being extracted from

  return tmdb_movie_ids, total_pages, total_results, current_page

In [8]:
def collect_month_data(year, month):
  """Job is to coordinate fetch_page_data and extract key data functions in order to
  fetch and extract an entire months data
  """
  data = fetch_page_data(year, month, 1) # fetches the first pages data in the search range

  tmdb_movie_ids, total_pages, total_results, current_page = extract_key_data(data) #takes the fetched data and stores the key data I need

  print(f""" 
  {calendar.month_name[month]}, {year} has {total_results} results and {total_pages} pages

  {len(tmdb_movie_ids)} movie ids collected as of {calendar.month_name[month]}, {year}, page {current_page}
  """)
  # now that I have fetched and extracted the data I need, I can collect each page for the month

  for page in range(2, total_pages +1):
    data = fetch_page_data(year, month, page)

    new_movie_ids, _, _, current_page = extract_key_data(data) # I don't need to return total pages or results anymore

    tmdb_movie_ids.extend(new_movie_ids) 
    #new ids is stored in a list. To not have the new ids added to the existing list as another nested list, I have to use .extend and not append


    print(f"""
    {len(tmdb_movie_ids)} movie ids collected as of {calendar.month_name[month]}, {year}, page {current_page}
    """)

  collected_id_count = len(tmdb_movie_ids)
  unique_collected_id_count = len(set(tmdb_movie_ids))

  if collected_id_count != total_results: #if the number of ids in the list is different than the number of total results I got from the first page I need to print a warning
    count_difference = total_results - collected_id_count
    print(f'WARNING\nMovie ID differential of {count_difference} for {calendar.month_name[month]}, {year}\nExpected: {total_results}\nCollected: {collected_id_count}')
  if unique_collected_id_count!= collected_id_count: #If the number of unique ids I collect is diff from the number of ids in general, there are duplicates
    duplicate_count = collected_id_count - unique_collected_id_count
    print(f'WARNING\n {duplicate_count} duplicate movie ids found in {calendar.month_name[month]}, {year}')

  #These two if statements handle both missing and duplicate ids

  return tmdb_movie_ids


In [9]:
def collect_year_data(year):
  """This function has the job of collecting data from a whole year. It depends on the function to get monthly data"""
  tmdb_movie_ids = collect_month_data(year, 1)
  
  for month in range (2, 13): 
    #will always be 12 months so I can hard code a 12 month limit
    
    
    new_ids = collect_month_data(year, month)
    tmdb_movie_ids.extend(new_ids)

    
    print(f'{calendar.month_name[month]}, {year} complete')

  return tmdb_movie_ids

In [10]:
def collect_multi_year_data(start_year, end_year):
  """This function has the job of collecting data over multiple years. It depends of the function to get data for a single year"""
  tmdb_movie_ids = collect_year_data(start_year)
  print(f'{start_year} Complete')
  for year in range(start_year +1, end_year+1):

    new_ids = collect_year_data(year)
    tmdb_movie_ids.extend(new_ids)

    print(f'{year} Complete')
  return tmdb_movie_ids

### Testing fuctions by collecting 2000-25 tmdb ids

In [11]:
start_time = time.perf_counter()

tmdb_ids_2000_25 = collect_multi_year_data(2000,2025)

end_time = time.perf_counter()

elapsed_time = end_time - start_time

elapsed_minutes = elapsed_time / 60

print(
    f"\n25-Year Collection Complete!\n"
    f"Total ids Collected: {len(tmdb_ids_2000_25):,}\n"
    f'Unique movies Collected: {len(set(tmdb_ids_2000_25)):,}\n '
    f"Execution time: {elapsed_time:.2f} seconds\n"
    f"Execution time: {elapsed_minutes:.2f} minutes")

 
  January, 2000 has 1413 results and 71 pages

  20 movie ids collected as of January, 2000, page 1
  

    40 movie ids collected as of January, 2000, page 2
    

    60 movie ids collected as of January, 2000, page 3
    

    80 movie ids collected as of January, 2000, page 4
    

    100 movie ids collected as of January, 2000, page 5
    

    120 movie ids collected as of January, 2000, page 6
    

    140 movie ids collected as of January, 2000, page 7
    

    160 movie ids collected as of January, 2000, page 8
    

    180 movie ids collected as of January, 2000, page 9
    

    200 movie ids collected as of January, 2000, page 10
    

    220 movie ids collected as of January, 2000, page 11
    

    240 movie ids collected as of January, 2000, page 12
    

    260 movie ids collected as of January, 2000, page 13
    

    280 movie ids collected as of January, 2000, page 14
    

    300 movie ids collected as of January, 2000, page 15
    

    320 movie ids colle